# PyMongo CRUD and Ingestion

## Imports

Complete the imports needed for:

- file paths
- JSON handling
- pandas
- MongoDB client

In [51]:
#!pip install pymongo

In [52]:
from pathlib import Path
import json

import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

## Connection settings

Use the same MongoDB service from the previous practical class.

Credentials source for this notebook:
- `prj/docker-compose.yml`
- root user: `labuser`
- root password: `labpass`
- authentication database: `admin`
- recommended host in this notebook: `host.docker.internal`

If the connection fails with `Authentication failed`, first test the container manually with:
- `docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin`

If that command also fails, the running volume was probably initialized earlier with different credentials.

Important local environment note:
- if you already have another local `mongod` running on port `27017`, `localhost` may connect to that service instead of the class container
- for this reason, prefer `host.docker.internal` as the MongoDB host in the notebook configuration

Database to reuse:
- `university_mongo`

New collection to create for this class:
- `student_performance`

In [53]:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "localhost",
    "port": 27017,
    "database": "census_income_kdd"
}

RAW_DIR = '../raw/clean'
COLLECTION_NAME = 'census'

## Documentation references:

- MongoDB PyMongo docs: [MongoClient](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.13/connect/mongoclient/)
- MongoDB PyMongo docs: [Find](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/find/)
- MongoDB PyMongo docs: [Insert](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/insert/)
- MongoDB PyMongo docs: [Update](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/update/)
- MongoDB PyMongo docs: [Delete](https://www.mongodb.com/docs/languages/python/pymongo-driver/v4.10/crud/delete/)

- UCI dataset page: Student Performance: https://archive.ics.uci.edu/dataset/320/student+performance

## Step 1 - Create a database class

Model your solution with the same idea used in previous database exercises:

- store configuration in the constructor
- create a `connect()` method
- create a `close()` method
- provide a method to retrieve a collection

This class should focus only on connection management.

In [54]:
class MongoDatabase:
    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        #store configuration fields such as host, port, user, password, database
        self.client = None
        self.db = None
        self.config = db_config

    def connect(self):
        # 1. Create a MongoClient only if it does not exist yet
        if self.client is not None:
            return self.db
        
        # 2. Build the URI with authSource=admin
        uri = f"mongodb://{self.config['user']}:{self.config['password']}@{self.config['host']}:{self.config['port']}/{self.config['database']}?authSource=admin"   
        
        # 3. Set a small serverSelectionTimeoutMS for friendlier failures
        try:
            self.client = MongoClient(uri, serverSelectionTimeoutMS=1000)

            # 4. Select the target database
            self.db = self.client[self.config['database']]
            
            # 5. Test the connection with ping
            self.client.admin.command('ping')
            return self.db
       
        # 6. If authentication fails, raise a clear message mentioning prj/docker-compose.yml and the docker exec mongosh command
        except OperationFailure as e:
            self.client = None
            raise RuntimeError(
                "MongoDB authentication failed.\n"
                "Check credentials in Final-Project/docker-compose.yml.\n"
                "You can test manually with:\n"
                "docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin"
            ) from e
        except ServerSelectionTimeoutError as e:
            self.client = None
            raise ServerSelectionTimeoutError(
                "Could not connect to MongoDB server. "
                "Check if the container/service is running and accessible."
            ) from e

    def is_connected(self):
        # Return True if the client is available, otherwise False
        return self.client is not None

    def get_collection(self, collection_name):
        # Return the collection object from the selected database
        return self.db[collection_name]

    def close(self):
        # Close the client and reset internal references
        if self.client is not None:
            self.client.close()
        self.client = None
        self.db = None

## Step 2 - Create a CRUD executor class

This second class should focus on collection operations, not on connection setup.

Suggested responsibilities:

- select documents
- insert one document
- insert many documents for ingestion
- update one document
- delete one document
- count documents

In [55]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        # 1. Keep a reference to the database object
        self.db = database
        # 2. Obtain the collection from the database object
        self.collection = self.db.get_collection(collection_name)

    def select(self, filter_query=None, projection=None, limit=10):
        # Implement a simple find() wrapper
        cursor = self.collection.find(filter=filter_query, projection=projection).limit(limit)
        return list(cursor)

    def insert_one(self, document):
        # Insert a single document
        return self.collection.insert_one(document)
        
    def insert_many(self, documents):
        # Insert many documents for the ingestion step
        return self.collection.insert_many(documents)

    def update_one(self, filter_query, update_query):
        # Update a single document
        return self.collection.update_one(filter_query, update_query)

    def delete_one(self, filter_query):
        # Delete a single document
        return self.collection.delete_one(filter_query)
    
    def delete_many(self, filter_query=None):
        # delete many documents for ingestion step (avoid repeating documents)
        return self.collection.delete_many(filter_query or {})

    def count_documents(self, filter_query=None):
        # Count documents, defaulting to all documents
        if not filter_query or filter_query is None:
            filter_query = {}
        return self.collection.count_documents(filter_query)

## Step 3 - Instantiate and test the connection

At this point, you should:

1. instantiate the `MongoDatabase`
2. connect to the database
3. instantiate the `MongoExecutor` for the new collection
4. confirm that the connection is working

Recommended check:
- call `db.connect()`
- if it fails, verify the same credentials with `mongosh` inside the container first

In [56]:
# 1. Instantiate the database object
db_manager = MongoDatabase(MONGO_CONFIG)

# 2. Connect to MongoDB
db = db_manager.connect()

# 3. Instantiate the executor for 'census' collection
census_executor = MongoExecutor(database=db_manager, collection_name=COLLECTION_NAME)

# Print a small success message or a count
try:
    current_docs = census_executor.count_documents()
    print(current_docs)
except Exception as e:
    print(f" Connection established, but failed to communicate with collection: {e}")

299288


## Step 4 - Map a raw row into a MongoDB document

Instead of inserting the raw CSV row blindly, create a mapping function.


In [57]:
def map_external_record(record, dataset_name, row_number):
    """Transform one CSV row into one MongoDB document."""

    # 1. build a stable document structure for the new collection
    formatted_id = f"{dataset_name}-{str(row_number).zfill(5)}"

    # 2. keep the source metadata in the document
    document = {
        "_id": formatted_id,  
        "external_student_id": formatted_id,
        "metadata": {
            "source_dataset": dataset_name,
            "source_row": row_number
        },
        "data": {}  
    }

    # 3. cast numeric fields when appropriate
    def casting(value):
        try:
           return float(value)
        except (ValueError, TypeError):
            return str(value)
        
    document = {}

    for key, value in record.items():
        document[key] = casting(value)

    return document

## Step 5 - Create the ingestion function

The ingestion function should:

1. read one CSV file
2. optionally limit the number of rows
3. map each row to a document
4. insert the documents into `student_performance_raw`
5. return how many documents were inserted

In [58]:
def ingest_data(executor, csv_path, limit=None):
    """Read a CSV file and insert its mapped rows into MongoDB."""

    # 1. Load the CSV with pandas
    df = pd.read_csv(csv_path)
    
    # 2. Apply the optional limit
    if limit is not None:
        df = df.head(limit)

    # 3. Convert rows to mapped documents
    documents = []
    for index, row in df.iterrows():
        documents.append(map_external_record(row, df, index))
    
    # 4. Insert documents into MongoDB
    executor.insert_many(documents)

    # 5. Return the number of inserted documents
    return len(documents)


In [59]:
db = MongoDatabase(MONGO_CONFIG)
db.connect()

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True, authsource='admin', serverselectiontimeoutms=1000), 'census_income_kdd')

In [60]:
collection = db.get_collection(COLLECTION_NAME)
executor = MongoExecutor(db, COLLECTION_NAME)

In [61]:
db.client.admin.command('ping')
print("Connected to MongoDB")

Connected to MongoDB


In [62]:
print(executor.count_documents({}))

299288


In [ ]:
#executor.delete_many({})

DeleteResult({'n': 299288, 'ok': 1.0}, acknowledged=True)

In [69]:
csv = RAW_DIR + '/census_income_kdd_clean.csv'

In [ ]:
# 1. Ingest a small batch first, for example 5 or 10 rows
#ingest_data(executor, csv, 10)

# 2. Validate the collection count
print(executor.count_documents({}))

# 3. Inspect one sample document
print(executor.select({}))

# Ingest the full file(s)
#executor.delete_many({})
#ingest_data(executor, csv)

0
[]


299288

In [66]:
print(executor.count_documents({}))

299288


In [67]:
executor.select({},{'_id': 0, 'education':'high_school_graduate', 'marital_stat':'widowed'})

[{'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'},
 {'education': 'high_school_graduate', 'marital_stat': 'widowed'}]